In [1]:
import os
import sys
import numpy as np
import torch

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.config import *
from src.data_loader import load_data
from src.preprocess import preprocess_list
from src.embedder import load_phobert, embed_texts
from src.trainer import train_svm, evaluate
from src.cso_optimizer import optimize_cso
from src.embedding_cache import get_or_create_embeddings
from src.split_cache import get_or_create_split

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

c:\Users\nguye\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
texts, labels = load_data(DATA_PATH)
print("Samples:", len(texts))

Columns: ['title', 'label']
Loaded samples: 208000
Samples: 208000


In [3]:
texts = preprocess_list(texts)

In [4]:
tokenizer, model = load_phobert(PHOBERT_DIR)

Load model từ local...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1690.54it/s]


In [7]:
EMBEDDING_DIR = "D:\DATN_Kaggle\models\cache_embeddings"

X, y = get_or_create_embeddings(
    texts=texts,
    labels=labels,
    embed_fn=embed_texts,
    tokenizer=tokenizer,
    model=model,
    save_dir=EMBEDDING_DIR
)

Loaded cached embeddings:
   X: (208000, 768)
   y: (208000,)


In [9]:
SPLIT_DIR = "D:\DATN_Kaggle\models\cache_split"

X_train, X_test, y_train, y_test = get_or_create_split(
    X, y,
    save_dir=SPLIT_DIR
)

Loaded cached split


In [ ]:
best_C = optimize_cso(X_train, y_train)
print("Best C:", best_C)

[CSO] Using 100000 samples for optimization
logC=2.3164 | C=207.203106 -> f1_macro=0.7494
logC=0.6355 | C=4.320459 -> f1_macro=0.7506
logC=-1.3822 | C=0.041477 -> f1_macro=0.7379
logC=-1.8499 | C=0.014127 -> f1_macro=0.7262
logC=0.8652 | C=7.331508 -> f1_macro=0.7502
logC=3.6972 | C=4979.539057 -> f1_macro=0.7494
logC=0.8597 | C=7.238581 -> f1_macro=0.7502
logC=-1.8677 | C=0.013561 -> f1_macro=0.7258
logC=2.3269 | C=212.272698 -> f1_macro=0.7494
logC=-1.3916 | C=0.040589 -> f1_macro=0.7378
logC=-1.8179 | C=0.015209 -> f1_macro=0.7269


In [ ]:
# model = Pipeline([
#     # ("scaler", StandardScaler()),
#     ("norm", Normalizer()),
#     ("svm", LinearSVC(
#         C=best_C,
#         class_weight="balanced",
#         max_iter=20000
#     ))
# ])

# model.fit(X_train, y_train)

from src.trainer import train_svm, evaluate

model = train_svm(X_train, y_train, C=best_C)
evaluate(model, X_test, y_test)

In [ ]:
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy: {acc:.4f}")
print(f"F1-macro: {f1:.4f}")

In [ ]:
import joblib
os.makedirs("models", exist_ok=True)

joblib.dump(model, "models/final_model.joblib")

print("Saved final model")

In [ ]:
import json

with open("../src/label_map.json", "r", encoding="utf-8") as f:
    label_map = json.load(f)

label_map = {int(k): v for k, v in label_map.items()}

In [ ]:
# def predict_text(text, model, tokenizer, phobert):
#     from src.preprocess import preprocess_text
#     from src.embedder import embed_texts

#     text = preprocess_text(text)
#     emb = embed_texts([text], tokenizer, phobert)

#     pred = model.predict(emb)[0]
#     return label_map[int(pred)]

In [ ]:
import pandas as pd
import joblib
from tqdm import tqdm

def predict_csv(input_csv, output_csv, model, tokenizer, phobert):
    from src.preprocess import preprocess_text
    from src.embedder import embed_texts

    # Load data
    df = pd.read_csv(input_csv)

    # 2. Check cột
    if "title" not in df.columns:
        raise ValueError("CSV phải có cột 'title'")

    texts = df["title"].astype(str).tolist()

    # Preprocess
    texts = [preprocess_text(t) for t in texts]

    # Embed theo batch
    batch_size = 64
    all_preds = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]

        emb = embed_texts(batch, tokenizer, phobert)
        preds = model.predict(emb)

        all_preds.extend(preds)

    # Map label
    # df["pred_label"] = [label_map[int(p)] for p in all_preds]
    df["label"] = [int(p) for p in all_preds]
    df = df[["title", "label"]]

    # Save
    df.to_csv(output_csv, index=False, encoding="utf-8-sig")

    print(f"Saved to {output_csv}")

In [ ]:
# _, phobert = load_phobert(PHOBERT_DIR)

# model = joblib.load("models/final_model.joblib")

# predict_text(
#     "Việt Nam phát hiện nhiều ca nhiễm biến thể Omicron mới",
#     model,
#     tokenizer,
#     phobert
# )

In [ ]:
_, phobert = load_phobert(PHOBERT_DIR)
model = joblib.load("models/final_model.joblib")

predict_csv(
    input_csv="D:\DATN_Kaggle\data\data_test.csv",
    output_csv="D:\DATN_Kaggle\data\test.csv",
    model=model,
    tokenizer=tokenizer,
    phobert=phobert
)